<a href="https://colab.research.google.com/github/aakankshakadam97/bfsi-churn-risk-scoring/blob/main/04_genai_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Cell 1: Install & Setup — Groq API
# 05_genai_layer.ipynb | bfsi-churn-risk-scoring
# ============================================================

# --- Install ---
!pip install -q groq pandas


print("✅ Groq SDK installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.5 MB/s eta 0:00:00
✅ Groq SDK installed


In [2]:


# --- Imports ---
import os
import json
import time
import pandas as pd
from datetime import datetime
from groq import Groq

# --- API Key Setup (Colab Secrets) ---
# Store your key once in Colab: left sidebar -> key icon -> "Add new secret"
# Name it exactly GROQ_API_KEY, and toggle "Notebook access" ON for this notebook.
# Get a free key at https://console.groq.com/keys
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception as e:
    from getpass import getpass
    print(f"Could not load from Colab secrets ({e}); falling back to manual entry.")
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# --- Model config ---
# NOTE (Aug 2026): Groq deprecated the llama3-* and llama-3.1/3.3-* family from
# free/developer tier — they're Enterprise-only now. Current dev/free-tier
# production models are the OpenAI GPT-OSS models served on Groq's LPUs.
# Always re-check https://console.groq.com/docs/models if this breaks again.
MODEL_NAME = "openai/gpt-oss-120b"     # higher quality, good for strategy text
FALLBACK_MODEL = "openai/gpt-oss-20b"  # faster/cheaper fallback

# --- Fetch live model list (belt-and-suspenders against future deprecations) ---
def get_available_models():
    try:
        models = client.models.list()
        return {m.id for m in models.data}
    except Exception as e:
        print(f"⚠️ Could not fetch model list: {e}")
        return set()

available = get_available_models()
if available:
    if MODEL_NAME not in available:
        print(f"⚠️ {MODEL_NAME} not in active model list.")
    if FALLBACK_MODEL not in available:
        print(f"⚠️ {FALLBACK_MODEL} not in active model list.")

# --- Quick connectivity test ---
def test_groq_connection():
    global MODEL_NAME
    try:
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": "Reply with exactly: OK"}],
            max_tokens=5,
            temperature=0
        )
        print(f"✅ Groq API connected. Model: {MODEL_NAME}")
        print(f"   Test response: {resp.choices[0].message.content.strip()}")
        return True
    except Exception as e:
        print(f"❌ Connection failed on {MODEL_NAME}: {e}")
        print(f"   Retrying with fallback model: {FALLBACK_MODEL}")
        try:
            resp = client.chat.completions.create(
                model=FALLBACK_MODEL,
                messages=[{"role": "user", "content": "Reply with exactly: OK"}],
                max_tokens=5,
                temperature=0
            )
            print(f"✅ Fallback connected. Response: {resp.choices[0].message.content.strip()}")
            MODEL_NAME = FALLBACK_MODEL  # switch active model for the rest of the notebook
            return True
        except Exception as e2:
            print(f"❌ Fallback also failed: {e2}")
            print("   Run `print(get_available_models())` to see what's actually live.")
            return False

connection_ok = test_groq_connection()

# --- Output paths ---
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
JSON_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "retention_strategies.json")
TXT_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "retention_strategies.txt")
EXEC_SUMMARY_PATH = os.path.join(OUTPUT_DIR, "executive_summary.txt")

print(f"\nOutputs will be saved to: {OUTPUT_DIR}/")

✅ Groq API connected. Model: openai/gpt-oss-120b
   Test response: 

Outputs will be saved to: outputs/


In [4]:
# ============================================================
# Cell 2: Load Data & Select Representative Customers
# 04_genai_layer.ipynb | bfsi-churn-risk-scoring
# ============================================================

# --- Load risk scores ---
DATA_PATH = "/content/risk_scores_final.csv"

df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"\nRecommended action distribution:")
print(df['recommended_action'].value_counts())

# --- Define the 6 actionable categories (excludes 'No action needed') ---
# NOTE: 'Dedicated RM call within 24 hours' does not appear in this dataset —
# Ultra HNI Critical customers who also trip the anomaly flag get routed to
# 'Flag for algo trading review' instead. Worth revisiting the rule priority
# in notebook 04 later, but for the GenAI layer we work with what's here.
ACTIONABLE_CATEGORIES = [
    "Flag for algo trading review — separate model needed",
    "RM call within 48 hours",
    "RM call + personalized email",
    "Automated push notification + email campaign",
    "Targeted email campaign",
    "Watchlist — monitor for 2 weeks"
]

missing = [a for a in ACTIONABLE_CATEGORIES if a not in df['recommended_action'].unique()]
if missing:
    print(f"\n⚠️ Missing expected action(s): {missing}")
else:
    print(f"\n✅ All {len(ACTIONABLE_CATEGORIES)} actionable categories found in data.")

# --- Select 1 representative customer per action type ---
# Strategy: highest churn_probability within each category (most urgent case)
selected_customers = []

for action in ACTIONABLE_CATEGORIES:
    subset = df[df['recommended_action'] == action]
    if subset.empty:
        print(f"⚠️ No rows found for: {action}")
        continue
    top_customer = subset.sort_values('churn_probability', ascending=False).iloc[0]
    selected_customers.append(top_customer)

selected_df = pd.DataFrame(selected_customers).reset_index(drop=True)

print(f"\n{'='*70}")
print(f"Selected {len(selected_df)} representative customers (1 per action type)")
print(f"{'='*70}\n")

display_cols = ['customer_id', 'segment', 'risk_tier', 'churn_probability',
                 'total_trade_value', 'is_anomaly', 'anomaly_type', 'recommended_action']
display_cols = [c for c in display_cols if c in selected_df.columns]
print(selected_df[display_cols].to_string(index=False))

Loaded 10,000 rows, 10 columns

Recommended action distribution:
recommended_action
No action needed                                        7922
Automated push notification + email campaign             815
RM call + personalized email                             457
Flag for algo trading review — separate model needed     360
Targeted email campaign                                  227
RM call within 48 hours                                  167
Watchlist — monitor for 2 weeks                           52
Name: count, dtype: int64

✅ All 6 actionable categories found in data.

Selected 6 representative customers (1 per action type)

customer_id   segment risk_tier  churn_probability  total_trade_value  is_anomaly          anomaly_type                                   recommended_action
 CUST016758 Ultra HNI  Critical           1.000000        55650488.44           1 High frequency trader Flag for algo trading review — separate model needed
 CUST016489       HNI  Critical           1.0

In [5]:
# ============================================================
# Cell 3: Generate Retention Strategies via Groq
# 05_genai_layer.ipynb | bfsi-churn-risk-scoring
# ============================================================

# --- Prompt template ---
# Builds a grounded, structured prompt per customer using only fields
# present in risk_scores_final.csv — no invented customer detail.
def build_prompt(row):
    anomaly_note = ""
    if row.get('is_anomaly') == 1:
        anomaly_note = f"\n- Anomaly flag: TRUE (type: {row.get('anomaly_type', 'Unknown')}) — this customer's trading behavior was flagged as statistically unusual by an Isolation Forest model."

    prompt = f"""You are a retention strategy advisor for a stock trading platform's relationship management team.

Generate a specific, actionable retention strategy for the following customer based ONLY on the data provided. Do not invent details not given below.

CUSTOMER PROFILE:
- Customer ID: {row['customer_id']}
- Segment: {row['segment']}
- Risk Tier: {row['risk_tier']}
- Churn Probability: {row['churn_probability']:.1%}
- Total Trade Value: ₹{row['total_trade_value']:,.0f}
- Recommended System Action: {row['recommended_action']}{anomaly_note}

Provide your response in this exact structure:

1. RISK SUMMARY (1-2 sentences): Why this customer is flagged, in plain business language.
2. RETENTION STRATEGY (3-4 sentences): A specific, actionable approach the RM/team should take, tailored to this customer's segment and risk tier.
3. KEY TALKING POINTS (3 bullet points): What the RM should say or offer in outreach.
4. URGENCY & TIMELINE: When this action should happen and why.

Keep the tone professional and grounded in BFSI relationship management practice. Do not mention that you are an AI."""

    return prompt


# --- Groq call wrapper with retry logic ---
def call_groq(prompt, model=MODEL_NAME, max_retries=3):
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=600,
                temperature=0.4  # some variation, but stay grounded/professional
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "rate" in err_str.lower():
                wait = 5 * (attempt + 1)
                print(f"   Rate limited, waiting {wait}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            elif attempt < max_retries - 1:
                print(f"   Error: {e}. Retrying...")
                time.sleep(2)
            else:
                print(f"   Failed after {max_retries} attempts: {e}")
                return f"[GENERATION FAILED: {e}]"
    return "[GENERATION FAILED: max retries exceeded]"


# --- Generation loop ---
results = []

print(f"Generating retention strategies for {len(selected_df)} customers using {MODEL_NAME}...\n")

for idx, row in selected_df.iterrows():
    print(f"[{idx+1}/{len(selected_df)}] {row['customer_id']} — {row['recommended_action']}")

    prompt = build_prompt(row)
    strategy_text = call_groq(prompt)

    results.append({
        "customer_id": row['customer_id'],
        "segment": row['segment'],
        "risk_tier": row['risk_tier'],
        "churn_probability": round(float(row['churn_probability']), 4),
        "total_trade_value": float(row['total_trade_value']),
        "is_anomaly": int(row['is_anomaly']),
        "anomaly_type": row.get('anomaly_type', None),
        "recommended_action": row['recommended_action'],
        "generated_strategy": strategy_text,
        "model_used": MODEL_NAME,
        "generated_at": datetime.now().isoformat()
    })

    time.sleep(2)  # stay comfortably under free-tier rate limits

print(f"\n✅ Generated {len(results)} retention strategies.")

# --- Preview first result ---
print(f"\n{'='*70}")
print(f"SAMPLE OUTPUT — {results[0]['customer_id']}")
print(f"{'='*70}")
print(results[0]['generated_strategy'])

Generating retention strategies for 6 customers using openai/gpt-oss-120b...

[1/6] CUST016758 — Flag for algo trading review — separate model needed
[2/6] CUST016489 — RM call within 48 hours
[3/6] CUST045455 — RM call + personalized email
[4/6] CUST019651 — Automated push notification + email campaign
[5/6] CUST028687 — Targeted email campaign
[6/6] CUST036831 — Watchlist — monitor for 2 weeks

✅ Generated 6 retention strategies.

SAMPLE OUTPUT — CUST016758
**1. RISK SUMMARY**  
CUST016758 is an Ultra‑HNI client with a “Critical” risk tier and a churn probability of 100 %; the Isolation Forest model flagged their high‑frequency, unusually large trade patterns (₹55.65 M total) as anomalous, indicating potential dissatisfaction or intent to move to a more specialized algo‑trading solution.

**2. RETENTION STRATEGY**  
Deploy a dedicated Relationship Manager (RM) with expertise in algorithmic trading to conduct a high‑touch, data‑driven review of the client’s current trading workflow wi

In [6]:
# ============================================================
# Cell 4: Display All Results & Save to JSON / TXT
# 05_genai_layer.ipynb | bfsi-churn-risk-scoring
# ============================================================

# --- Display all 6 strategies ---
for i, r in enumerate(results, 1):
    print(f"\n{'='*80}")
    print(f"[{i}/6] {r['customer_id']} | {r['segment']} | {r['risk_tier']} | "
          f"Churn Prob: {r['churn_probability']:.1%}")
    print(f"Action: {r['recommended_action']}")
    if r['is_anomaly']:
        print(f"Anomaly: {r['anomaly_type']}")
    print(f"{'='*80}")
    print(r['generated_strategy'])

print(f"\n\n{'#'*80}")
print(f"# All {len(results)} strategies displayed above")
print(f"{'#'*80}")

# --- Save to JSON ---
with open(JSON_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n✅ Saved JSON: {JSON_OUTPUT_PATH}")

# --- Save to TXT (human-readable report format) ---
with open(TXT_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("RETENTION STRATEGIES REPORT\n")
    f.write("bfsi-churn-risk-scoring | GenAI Action Layer\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model: {MODEL_NAME}\n")
    f.write("="*80 + "\n\n")

    for i, r in enumerate(results, 1):
        f.write(f"[{i}/{len(results)}] CUSTOMER: {r['customer_id']}\n")
        f.write(f"Segment: {r['segment']} | Risk Tier: {r['risk_tier']} | "
                f"Churn Probability: {r['churn_probability']:.1%}\n")
        f.write(f"Total Trade Value: ₹{r['total_trade_value']:,.0f}\n")
        f.write(f"Recommended System Action: {r['recommended_action']}\n")
        if r['is_anomaly']:
            f.write(f"Anomaly Flag: {r['anomaly_type']}\n")
        f.write("-"*80 + "\n")
        f.write(r['generated_strategy'] + "\n")
        f.write("="*80 + "\n\n")

print(f"✅ Saved TXT: {TXT_OUTPUT_PATH}")

# --- Quick file size check ---
for path in [JSON_OUTPUT_PATH, TXT_OUTPUT_PATH]:
    size_kb = os.path.getsize(path) / 1024
    print(f"   {path}: {size_kb:.1f} KB")


[1/6] CUST016758 | Ultra HNI | Critical | Churn Prob: 100.0%
Action: Flag for algo trading review — separate model needed
Anomaly: High frequency trader
**1. RISK SUMMARY**  
CUST016758 is an Ultra‑HNI client with a “Critical” risk tier and a churn probability of 100 %; the Isolation Forest model flagged their high‑frequency, unusually large trade patterns (₹55.65 M total) as anomalous, indicating potential dissatisfaction or intent to move to a more specialized algo‑trading solution.

**2. RETENTION STRATEGY**  
Deploy a dedicated Relationship Manager (RM) with expertise in algorithmic trading to conduct a high‑touch, data‑driven review of the client’s current trading workflow within 24 hours. Offer a bespoke “Algo‑Edge” package that includes a custom‑built execution engine, lower transaction fees for high‑volume trades, and direct access to a senior quant‑analyst for strategy refinement. Simultaneously, schedule a joint session with the platform’s technology team to co‑design a tail

#Gen AI action layer
**Purpose:** Translate quantitative risk scores (churn probability, risk tier, anomaly flags) into human-readable retention strategies and a leadership-facing executive summary, using an LLM as a synthesis layer on top of deterministic model outputs — not as a replacement for them.

**Inputs:** `risk_scores_final.csv` (10,000 customers), output of notebooks 01–04 (EDA → Churn Model → Risk Scoring).

**Approach:**
1. Load scored customer data and identify the 6 actionable `recommended_action` categories present in the data (excludes the ~79% of the base requiring no action).
2. Select one representative customer per action category — the highest-churn-probability case in each — as a deep-dive example.
3. Generate a grounded retention strategy per customer via Groq (`openai/gpt-oss-120b`), using only fields present in the dataset — no fabricated customer detail.
4. Generate an executive summary for leadership, where all quantitative claims (churn %, trade value at risk, segment distribution) are computed in pandas and passed into the prompt as ground truth; the LLM is used for narrative synthesis and prioritization, not calculation.

**Design note:** A known gap surfaced during this step — the "Dedicated RM call within 24 hours" action (intended for Ultra HNI Critical customers) never appears in the output data, because the anomaly-detection rule in notebook 04 takes priority over the segment/tier rule whenever both conditions are met. This is flagged here rather than silently worked around, and is a candidate fix for the risk-scoring rule logic in a future iteration.

**Outputs:**
- `outputs/retention_strategies.json` — structured, machine-readable strategy records
- `outputs/retention_strategies.txt` — human-readable report
- `outputs/executive_summary.txt` — leadership briefing with supporting portfolio stats

**Model:** Groq API, `openai/gpt-oss-120b` (free tier). Note: Groq deprecated the `llama3-*` model family from the free tier in 2026 — model selection in Cell 1 includes a live availability check against `client.models.list()` to avoid silent failures if this happens again.

In [7]:
# ============================================================
# Cell 5: Executive Summary for Leadership
# 05_genai_layer.ipynb | bfsi-churn-risk-scoring
# ============================================================

# --- Compute portfolio-level stats from the FULL dataset (not just the 6 samples) ---
total_customers = len(df)
critical_count = (df['risk_tier'] == 'Critical').sum()
high_count = (df['risk_tier'] == 'High').sum()
anomaly_count = (df['is_anomaly'] == 1).sum()

action_counts = df['recommended_action'].value_counts().to_dict()

# Segment x risk_tier breakdown for Critical/High only (leadership cares about this)
urgent_df = df[df['risk_tier'].isin(['Critical', 'High'])]
segment_urgency = urgent_df.groupby(['segment', 'risk_tier']).size().unstack(fill_value=0)

# Trade value at risk (Critical tier only)
critical_trade_value = df[df['risk_tier'] == 'Critical']['total_trade_value'].sum()

portfolio_stats = f"""
PORTFOLIO OVERVIEW:
- Total customers analyzed: {total_customers:,}
- Critical risk tier: {critical_count:,} ({critical_count/total_customers:.1%})
- High risk tier: {high_count:,} ({high_count/total_customers:.1%})
- Anomalous trading behavior flagged: {anomaly_count:,} ({anomaly_count/total_customers:.1%})
- Total trade value at risk (Critical tier): ₹{critical_trade_value:,.0f}

ACTION DISTRIBUTION:
{chr(10).join(f'- {k}: {v:,}' for k, v in action_counts.items())}

SEGMENT x RISK TIER (Critical/High only):
{segment_urgency.to_string()}
"""

print(portfolio_stats)

# --- Build executive summary prompt ---
exec_prompt = f"""You are preparing a briefing for senior leadership (CXO level) at a stock trading platform, based on a churn risk scoring and anomaly detection exercise covering the full customer base.

Below is the portfolio-level data:
{portfolio_stats}

Also, here are 6 representative individual cases that were deep-dived (one per recommended action category):
{chr(10).join(f"- {r['customer_id']} ({r['segment']}, {r['risk_tier']}, {r['churn_probability']:.0%} churn risk): {r['recommended_action']}" for r in results)}

Write an executive summary for leadership with this structure:

1. HEADLINE (1-2 sentences): The single most important takeaway.
2. KEY FINDINGS (3-4 bullet points): What the data shows, grounded in the numbers above. Do not invent figures not provided.
3. BUSINESS IMPACT: Financial/relationship risk implications, referencing trade value and segment concentration where relevant.
4. RECOMMENDED ACTIONS (3-4 bullet points): Concrete next steps for the RM/leadership team, prioritized.
5. MODEL NOTE (1-2 sentences): Brief mention that this is powered by a Random Forest churn model (ROC-AUC 0.98) and Isolation Forest anomaly detection, for credibility/transparency.

Keep it concise, business-toned, and free of technical jargon beyond what's necessary. Target length: 300-400 words. Do not mention that you are an AI."""

print("Generating executive summary...\n")
exec_summary = call_groq(exec_prompt, max_retries=3)

print("="*80)
print("EXECUTIVE SUMMARY")
print("="*80)
print(exec_summary)

# --- Save executive summary ---
with open(EXEC_SUMMARY_PATH, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("EXECUTIVE SUMMARY — CHURN RISK & RETENTION\n")
    f.write("bfsi-churn-risk-scoring | GenAI Action Layer\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model: {MODEL_NAME}\n")
    f.write("="*80 + "\n\n")
    f.write(exec_summary + "\n\n")
    f.write("-"*80 + "\n")
    f.write("SUPPORTING PORTFOLIO DATA\n")
    f.write("-"*80 + "\n")
    f.write(portfolio_stats)

print(f"\n✅ Saved: {EXEC_SUMMARY_PATH}")
print(f"   Size: {os.path.getsize(EXEC_SUMMARY_PATH)/1024:.1f} KB")


PORTFOLIO OVERVIEW:
- Total customers analyzed: 10,000
- Critical risk tier: 1,521 (15.2%)
- High risk tier: 227 (2.3%)
- Anomalous trading behavior flagged: 500 (5.0%)
- Total trade value at risk (Critical tier): ₹15,910,671,868

ACTION DISTRIBUTION:
- No action needed: 7,922
- Automated push notification + email campaign: 815
- RM call + personalized email: 457
- Flag for algo trading review — separate model needed: 360
- Targeted email campaign: 227
- RM call within 48 hours: 167
- Watchlist — monitor for 2 weeks: 52

SEGMENT x RISK TIER (Critical/High only):
risk_tier  Critical  High
segment                  
Affluent        457     0
HNI             203     0
Mass            815   227
Ultra HNI        46     0

Generating executive summary...

EXECUTIVE SUMMARY
**HEADLINE**  
Nearly 15 % of our active client base (1,521 accounts) is classified as critical‑risk, exposing ₹15.9 billion of trade value to potential churn or adverse trading anomalies.

**KEY FINDINGS**  
- **Critical‑